# TP Jour 4 — Tests d'Hypothèses Statistiques, Segmentation & Restitution BI


In [32]:
import numpy as np
import pandas as pd
from scipy import stats
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = (
    SparkSession.builder
    .appName("J4_Stats_Segmentation_Dashboard")
    .getOrCreate()
)

## 1. Inférence Statistique : Tester vos Hypothèses Métier (`scipy.stats`)

Choisissez **au moins deux tests statistiques** pertinents pour votre sujet parmi :
- **Comparaison de 2 groupes indépendants** sur une variable asymétrique : **Mann-Whitney U** (`stats.mannwhitneyu`).
- **Comparaison de $k \ge 3$ groupes** : **Kruskal-Wallis** (`stats.kruskal`).
- **Lien monotone entre 2 variables quantitatives** non gaussiennes : **Corrélation de rangs de Spearman** (`stats.spearmanr`).
- **Dépendance entre 2 variables qualitatives** : **Test du $\chi^2$** (`stats.chi2_contingency`).

### Exercice 1.1 — Formuler $H_0 / H_1$, projeter les colonnes utiles depuis Silver et conclure
> **Bonne pratique Big Data :** Ne faites jamais `.toPandas()` sur toute une table Silver brute ! Filtrez et sélectionnez (`select(...)`) uniquement les colonnes nécessaires côté PySpark avant de rapatrier les données vers le driver.

**Formulation des hypothèses :**

**Test 1 — Mann-Whitney U**

- H0 : les retards à l'arrivée des compagnies AA et DL suivent la même distribution.
- H1 : les distributions des retards à l'arrivée sont différentes.

Le test de Mann-Whitney est utilisé pour comparer deux groupes indépendants. Il convient ici car la distribution des retards est asymétrique.

**Test 2 — Corrélation de Spearman**

- H0 : il n'existe pas de corrélation monotone entre les retards au départ et à l'arrivée.
- H1 : il existe une corrélation monotone entre ces deux variables.

Le test de Spearman permet d'étudier la relation entre deux variables quantitatives sans supposer qu'elles suivent une distribution normale.

In [33]:
SILVER_MAIN_PATH = "/lakehouse/silver/flights_enriched"

# Sélection des colonnes utiles et échantillonnage avant Pandas.
df_sample = (
    spark.read.format("delta").load(SILVER_MAIN_PATH)
    .select("airline", "departure_delay", "arrival_delay")
    .dropna()
    .sample(False, 0.02, seed=42)
    .toPandas()
)

assert df_sample is not None, "TODO : Extrayez les colonnes nécessaires dans `df_sample`."

# Comparaison des retards à l'arrivée entre AA et DL.
groupe_a = df_sample[df_sample["airline"] == "AA"]["arrival_delay"]
groupe_b = df_sample[df_sample["airline"] == "DL"]["arrival_delay"]

stat_1, p_val_1 = stats.mannwhitneyu(
    groupe_a, groupe_b, alternative="two-sided"
)

print("Mann-Whitney U :", stat_1, "| p-value :", p_val_1)
print("Conclusion :", "H0 rejetée" if p_val_1 < 0.05 else "H0 non rejetée")

# Recherche d'une relation entre les retards au départ et à l'arrivée.
rho, p_val_2 = stats.spearmanr(
    df_sample["departure_delay"],
    df_sample["arrival_delay"]
)

print("Spearman :", rho, "| p-value :", p_val_2)
print("Conclusion :", "H0 rejetée" if p_val_2 < 0.05 else "H0 non rejetée")

/opt/venv/lib/python3.15/site-packages/pyspark/sql/pandas/conversion.py:298: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


Mann-Whitney U : 130164849.5 | p-value : 7.928264259877259e-16
Conclusion : H0 rejetée
Spearman : 0.6381242039468682 | p-value : 0.0
Conclusion : H0 rejetée


### Résultats des tests statistiques

Les tests ont été réalisés sur un échantillon de 2 % des vols, après sélection des colonnes nécessaires dans la table Silver.

Le test de Mann-Whitney donne une statistique de **130 164 849,5** et une p-value de **7,93 × 10⁻¹⁶**. Cette valeur étant inférieure au seuil de 5 %, l'hypothèse nulle est rejetée. Une différence statistiquement significative est donc observée entre les distributions des retards à l'arrivée des compagnies AA et DL.

Pour le test de Spearman, le coefficient obtenu est de **0,638**, avec une p-value affichée à 0 en raison de sa très faible valeur. L'hypothèse nulle est également rejetée. Il existe une corrélation positive assez marquée entre les retards au départ et à l'arrivée. Les vols qui partent en retard ont donc généralement tendance à arriver en retard.

Ces résultats montrent que les retards varient selon les compagnies étudiées et qu'un retard au départ est associé à un retard à l'arrivée.

## 2. Segmentation / Typologie par Clustering K-Means (`/lakehouse/gold/`)

Quelle que soit la thématique de votre dataset Kaggle, identifiez l'**entité métier pertinente à segmenter** (ex. clients/utilisateurs, stations, zones géographiques, produits, lignes de transport, capteurs...) :
1. **Agrégation PySpark** : résumez chaque entité par **3 à 5 features numériques comportementales** (fréquence, volume, valeur médiane/totale, récence, taux d'incident...).
2. **Prétraitement statistique** : appliquez `np.log1p(...)` sur les features positives asymétriques puis centrez-réduisez avec `StandardScaler()` afin qu'aucune variable à grande échelle n'écrase la distance euclidienne du K-Means.
3. **Interprétation métier** : inspectez les médianes par cluster (`groupby("cluster_id").median()`), attribuez un **nom explicite** à chaque segment (`colonne "segment"`), puis enregistrez la table dans **`/lakehouse/gold/<nom_table_segments>`**.

In [34]:
GOLD_SEGMENTS_PATH = "/lakehouse/gold/airline_segments"

# TODO 1 : Agréger les features par entité métier côté PySpark puis convertir la table d'entités en Pandas

# Calcul des indicateurs pour chaque compagnie.
entities_spark = (
    spark.read.format("delta").load("/lakehouse/silver/flights_enriched")
    .filter(F.col("airline").isNotNull())
    .groupBy("airline")
    .agg(
        F.count("*").alias("total_flights"),
        F.avg("arrival_delay").alias("avg_arrival_delay"),
        F.expr("percentile_approx(arrival_delay, 0.90)").alias("p90_arrival_delay"),
        F.avg(
            F.when(F.col("cancelled") == 1, 1.0).otherwise(0.0)
        ).alias("cancellation_rate")
    )
)

entities_pd = entities_spark.toPandas()

assert entities_pd is not None, "TODO : Construisez le DataFrame d'entités agrégées `entities_pd`."

# TODO 2 : Prétraiter (log1p + StandardScaler), entraîner KMeans et nommer métier chaque cluster dans `entities_pd["segment"]`

features = [
    "total_flights",
    "avg_arrival_delay",
    "p90_arrival_delay",
    "cancellation_rate"
]

X = entities_pd[features].fillna(0).copy()

# Réduction de l'asymétrie des variables positives.
X["total_flights"] = np.log1p(X["total_flights"])
X["p90_arrival_delay"] = np.log1p(
    X["p90_arrival_delay"].clip(lower=0)
)

# Standardisation pour comparer les variables.
X_scaled = StandardScaler().fit_transform(X)

# Création de trois groupes de compagnies.
kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
entities_pd["cluster_id"] = kmeans.fit_predict(X_scaled)

# Comparaison des médianes pour interpréter les groupes.
cluster_summary = entities_pd.groupby("cluster_id")[features].median()

print("Médianes par cluster :")
print(cluster_summary)

# Attribution des noms selon le volume de vols.
cluster_order = (
    cluster_summary["total_flights"]
    .sort_values()
    .index
    .tolist()
)

segment_names = {
    cluster_order[0]: "Faible volume",
    cluster_order[1]: "Volume intermédiaire",
    cluster_order[2]: "Fort volume"
}

entities_pd["segment"] = entities_pd["cluster_id"].map(segment_names)

assert "segment" in entities_pd.columns, "TODO : Ajoutez une colonne lisible `segment` décrivant chaque cluster."

# TODO 3 : Sauvegarder en Delta Lake sur HDFS dans `GOLD_SEGMENTS_PATH`
(
    spark.createDataFrame(entities_pd)
    .coalesce(2)
    .write.format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .save(GOLD_SEGMENTS_PATH)
)

print(
    f"Table de segmentation enregistrée dans {GOLD_SEGMENTS_PATH} "
    f"({len(entities_pd):,} entités) !"
)

/opt/venv/lib/python3.15/site-packages/pyspark/sql/pandas/conversion.py:298: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


Médianes par cluster :
            total_flights  avg_arrival_delay  p90_arrival_delay  \
cluster_id                                                        
0                 90836.0          12.504706               54.0   
1                543850.0           5.638623               37.5   
2                172521.0           0.186754               20.0   

            cancellation_rate  
cluster_id                     
0                    0.008626  
1                    0.016470  
2                    0.003878  


/opt/venv/lib/python3.15/site-packages/pyspark/sql/pandas/conversion.py:659: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()
/opt/venv/lib/python3.15/site-packages/pyspark/sql/pandas/conversion.py:936: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


Table de segmentation enregistrée dans /lakehouse/gold/airline_segments (14 entités) !


### Résultats de la segmentation K-Means

La segmentation a été réalisée sur les **14 compagnies aériennes**, en utilisant quatre indicateurs : le nombre de vols, le retard moyen à l'arrivée, le 90e percentile des retards et le taux d'annulation. Après transformation et standardisation des données, l'algorithme K-Means a permis de former trois groupes.

Le premier groupe, nommé **Faible volume**, présente une médiane de 90 836 vols. Il possède les retards les plus élevés, avec un retard moyen médian de 12,50 minutes et un P90 de 54 minutes.

Le deuxième groupe, nommé **Fort volume**, présente une médiane de 543 850 vols. Son retard moyen médian est de 5,64 minutes et son taux d'annulation médian atteint 1,65 %, le plus élevé des trois groupes.

Le troisième groupe, nommé **Volume intermédiaire**, présente une médiane de 172 521 vols. Il affiche les retards les plus faibles, avec un retard moyen médian de 0,19 minute et un P90 de 20 minutes. Son taux d'annulation médian est également le plus faible, à environ 0,39 %.

Cette segmentation montre que les compagnies ne se distinguent pas uniquement par leur volume de vols, mais également par leurs performances en matière de ponctualité. Le groupe de volume intermédiaire présente les meilleurs résultats sur les indicateurs de retard et d'annulation étudiés.

Les résultats ont ensuite été enregistrés dans la table Delta `airline_segments` de la couche Gold.

## 3. Livrable Final : Votre Application `app/dashboard.py` sur Streamlit (`:8501`)

Complétez maintenant **`app/dashboard.py`** pour valoriser vos tables `/lakehouse/gold/` :
- **Architecture respectée** : l'application Streamlit ne doit lire **que** des tables pré-calculées de la couche **Gold** (`/lakehouse/gold/...`), jamais les fichiers CSV bruts !
- **Interactivité** : au moins un filtre utilisateur (`st.selectbox`, `st.multiselect`, `st.slider`) et 3 à 4 visualisations Plotly Express complémentaires.
- **Rigueur explicative** : sous chaque visualisation clé, intégrez un `st.expander("Requête SQL & Justification statistique")` affichant la requête Spark SQL ayant produit la table Gold et justifiant le choix des indicateurs statistiques retenus.

In [35]:
# Vérification finale de l'inventaire complet de votre couche Gold sur HDFS
jvm = spark._jvm
fs = jvm.org.apache.hadoop.fs.FileSystem.get(spark._jsc.hadoopConfiguration())
PathHDFS = jvm.org.apache.hadoop.fs.Path
gold_root = PathHDFS("/lakehouse/gold")

gold_tables = [
    s.getPath().getName()
    for s in fs.listStatus(gold_root)
    if s.isDirectory() and fs.exists(PathHDFS(s.getPath(), "_delta_log"))
]
assert len(gold_tables) > 0, "Aucune table Delta trouvée dans /lakehouse/gold/ !"

print("=== Tables Delta disponibles dans /lakehouse/gold/ pour Streamlit ===")
for t in sorted(gold_tables):
    cnt = spark.read.format("delta").load(f"/lakehouse/gold/{t}").count()
    print(f" - /lakehouse/gold/{t:<25} : {cnt:>9,} lignes")

=== Tables Delta disponibles dans /lakehouse/gold/ pour Streamlit ===
 - /lakehouse/gold/airline_delay_pareto      :        14 lignes
 - /lakehouse/gold/airline_segments          :        14 lignes
 - /lakehouse/gold/monthly_flight_trends     :        12 lignes
 - /lakehouse/gold/top_airlines_by_origin    :       628 lignes



### 3.1. Présentation du dashboard

Une application Streamlit a été développée pour rassembler les résultats des différentes analyses dans une interface interactive. Elle permet de consulter les indicateurs principaux et de comparer plus facilement les compagnies aériennes.

L'application récupère les données directement depuis les tables Delta de la couche Gold. Les agrégations et les calculs ont déjà été effectués avec Spark, ce qui évite de les répéter à chaque affichage.

Quatre tables sont utilisées :

| Table Gold | Données exploitées |
|---|---|
| `airline_delay_pareto` | Retards cumulés par compagnie |
| `monthly_flight_trends` | Nombre de vols et évolution mensuelle |
| `airline_segments` | Caractéristiques et segments des compagnies |
| `top_airlines_by_origin` | Compagnies principales par aéroport |

Les tables sont chargées avec PySpark, puis converties en DataFrames Pandas pour construire les graphiques avec Plotly.

La session Spark est conservée en cache pour éviter de la recréer à chaque interaction. Les données chargées sont également mises en cache afin de réduire les temps d'attente lors de la navigation.

### 3.2. Analyse des retards

La première visualisation présente le total des minutes de retard par compagnie, sous la forme d'un diagramme en barres.

Les compagnies sont classées par ordre décroissant, ce qui permet de repérer rapidement celles qui cumulent le plus de retard.

Une courbe de Pareto complète le graphique en représentant la contribution cumulée des compagnies au total des retards. Elle permet d'observer si les retards sont concentrés sur un petit nombre de compagnies ou répartis de manière plus homogène.

Il faut cependant tenir compte du nombre de vols : une compagnie très active peut cumuler davantage de minutes de retard simplement parce qu'elle effectue plus de trajets.

### 3.3. Évolution mensuelle du trafic

Le deuxième graphique représente l'évolution du nombre de vols au fil des mois.

La courbe permet de visualiser les variations du trafic et d'identifier les périodes où le volume de vols est plus élevé ou plus faible.

Un graphique complémentaire présente le taux de variation mensuel. Celui-ci est calculé en comparant le nombre de vols du mois courant à celui du mois précédent.

Cette représentation facilite la lecture des tendances temporelles, même si elle ne permet pas d'expliquer directement les raisons des variations observées.

### 3.4. Visualisation des segments K-Means

Les résultats de la segmentation sont présentés dans un nuage de points. Chaque point correspond à une compagnie aérienne.

L'axe horizontal représente le nombre de vols, tandis que l'axe vertical indique le retard moyen à l'arrivée. Les couleurs permettent de distinguer les trois groupes obtenus avec K-Means.

La segmentation repose sur quatre caractéristiques : le volume de vols, le retard moyen, le 90e percentile du retard et le taux d'annulation.

Le graphique permet de comparer visuellement les compagnies, mais il ne représente directement que deux des quatre variables utilisées pour former les groupes.

Les résultats montrent trois profils distincts : les compagnies à faible volume, celles à fort volume et celles à volume intermédiaire.

Dans les résultats obtenus, le groupe à faible volume présente les retards moyens les plus élevés. Le groupe à fort volume rassemble les compagnies ayant l'activité la plus importante, tandis que le groupe à volume intermédiaire présente les retards moyens les plus faibles.

Ces différences permettent de mieux caractériser les compagnies, sans considérer les segments comme un classement définitif de leur performance.

### 3.5. Compagnies principales par aéroport

La dernière visualisation présente les compagnies qui réalisent le plus grand nombre de vols au départ de chaque aéroport.

Le graphique en barres horizontales facilite la comparaison des volumes. Les aéroports sont classés selon le nombre de vols effectués par leur compagnie principale.

Le classement a été calculé avec la fonction SQL `ROW_NUMBER()`, qui permet d'attribuer un rang aux compagnies pour chaque aéroport.

Le graphique présente uniquement les compagnies arrivées en première position. Il ne représente donc pas l'ensemble des vols réalisés dans chaque aéroport.

### 3.6. Filtres et navigation

Une barre latérale permet de sélectionner les compagnies à afficher et de filtrer les résultats de la segmentation.

Le nombre d'aéroports présentés dans le classement peut également être modifié afin de conserver un affichage lisible.

Les différentes analyses sont réparties dans plusieurs onglets. Des indicateurs synthétiques sont affichés en haut de l'application pour donner une vue d'ensemble des données disponibles.

Les graphiques sont interactifs : il est possible de survoler les éléments pour consulter leurs valeurs. Des tableaux détaillés et des sections contenant les requêtes SQL permettent également d'accéder aux informations complémentaires.

L'interface utilise un fond sombre avec des accents roses et lavande. Les cartes arrondies et la séparation des analyses en onglets facilitent la lecture et la navigation.

### 3.7. Bilan

Le dashboard permet de consulter les résultats des analyses dans une seule application, sans avoir à exécuter individuellement les requêtes SQL ou les traitements de segmentation.

Les graphiques mettent en évidence plusieurs aspects des données : la répartition des retards entre les compagnies, l'évolution du trafic, les différences entre les profils K-Means et la présence des compagnies dans les aéroports.

Les filtres permettent d'explorer les résultats plus précisément, tandis que les tableaux et les requêtes SQL apportent des informations complémentaires.

L'application complète ainsi les analyses statistiques et la segmentation en proposant une restitution visuelle des résultats obtenus.
